# Clipt — PARSeq Jersey OCR Fine-Tuning

Fine-tunes scene text recognition to detect jersey #2 in football footage.

Based on Koshkina et al. CVPR 2024 (91.4% accuracy on hockey).

**Step 2 of 2:** Run `clipt_upscaling.ipynb` first to get `CLIP_URLS`.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU — change runtime to A100')

In [ ]:
import subprocess, os

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    status = 'OK' if result.returncode == 0 else 'ERROR'
    print(f'{status}: {cmd[:70]}')
    if result.returncode != 0:
        print(result.stderr[-300:])
    return result.returncode == 0

# Install YOLOv8 for player detection
run('pip install -q ultralytics easyocr')

# Install PARSeq dependencies
run('pip install -q torch torchvision timm lmdb Pillow')

# Clone PARSeq (the actual model, not Koshkina wrapper)
# Koshkina pipeline uses PARSeq underneath but the wrapper has
# hardcoded dataset names — we use PARSeq directly
if not os.path.exists('/content/parseq'):
    run('git clone https://github.com/baudm/parseq.git /content/parseq')

run('pip install -q -r /content/parseq/requirements/train.txt', cwd='/content/parseq')

# Install cloudinary for downloading clips
run('pip install -q cloudinary requests')

print('All dependencies installed')

In [ ]:
# ============================================================
# EDIT THIS CELL
# ============================================================

# Paste the CLIP_URLS output from clipt_upscaling.ipynb here:
CLIP_URLS = [
    # Example — replace with actual Cloudinary URLs from upscaling notebook:
    # "https://res.cloudinary.com/dc33vjyyv/video/upload/v.../clipt-clips-720p/clip_1_1686s_720p.mp4",
]

# Target jersey number to train on
TARGET_JERSEY = '2'

# Training settings (good defaults for A100)
EPOCHS = 15
BATCH_SIZE = 32

# ============================================================
# DO NOT EDIT BELOW
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/crops/positive', exist_ok=True)  # Jersey #2 crops
os.makedirs('/content/crops/negative', exist_ok=True)  # Other jersey crops
os.makedirs('/content/lmdb_train', exist_ok=True)
os.makedirs('/content/lmdb_val', exist_ok=True)

print(f'Training jersey #{TARGET_JERSEY} detector')
print(f'Input clips: {len(CLIP_URLS)}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}')
if not CLIP_URLS:
    print('WARNING: CLIP_URLS is empty — run clipt_upscaling.ipynb first')

In [ ]:
import cv2, requests, os, numpy as np
from ultralytics import YOLO

# Load YOLOv8 for person detection
model = YOLO('yolov8n.pt')

crop_count = 0
label_lines = []

for clip_idx, url in enumerate(CLIP_URLS):
    clip_path = f'/content/clip_{clip_idx}.mp4'

    print(f'\nDownloading clip {clip_idx+1}/{len(CLIP_URLS)}...')
    r = requests.get(url, stream=True, timeout=300)
    with open(clip_path, 'wb') as f:
        for chunk in r.iter_content(1024*1024):
            f.write(chunk)

    cap = cv2.VideoCapture(clip_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'  {w}x{h} {total_frames} frames at {fps:.0f}fps')

    # Sample every 10 frames
    sample_interval = max(1, int(fps / 3))
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_interval == 0:
            # Detect players with YOLOv8
            results = model(frame, classes=[0], conf=0.3, verbose=False)

            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    bh = y2 - y1
                    bw = x2 - x1

                    if bh < 30:  # Too small
                        continue

                    # Crop torso region (where jersey number is)
                    torso_y1 = y1 + int(bh * 0.2)
                    torso_y2 = y1 + int(bh * 0.55)
                    torso_x1 = x1 + int(bw * 0.15)
                    torso_x2 = x2 - int(bw * 0.15)

                    crop = frame[torso_y1:torso_y2, torso_x1:torso_x2]
                    if crop.size == 0 or crop.shape[0] < 15 or crop.shape[1] < 10:
                        continue

                    # Upscale small crops for better OCR
                    if crop.shape[0] < 64:
                        scale = 64 / crop.shape[0]
                        crop = cv2.resize(crop, (int(crop.shape[1]*scale), 64),
                                         interpolation=cv2.INTER_LANCZOS4)

                    crop_name = f'clip{clip_idx}_frame{frame_idx}_det{crop_count}.jpg'
                    crop_path = f'/content/crops/positive/{crop_name}'
                    cv2.imwrite(crop_path, crop)
                    label_lines.append(f'{crop_name}\t{TARGET_JERSEY}')
                    crop_count += 1

        frame_idx += 1

    cap.release()
    print(f'  Extracted {crop_count} total crops so far')

# Save labels
with open('/content/crops/labels.txt', 'w') as f:
    f.write('\n'.join(label_lines))

print(f'\nTotal crops: {crop_count}')
print(f'Labels saved to /content/crops/labels.txt')
if crop_count < 100:
    print('WARNING: fewer than 100 crops — results may be poor. Run more clips through upscaling notebook.')
elif crop_count < 500:
    print('NOTE: 500+ crops recommended for best results. Current count is marginal.')
else:
    print('Crop count looks good for fine-tuning')

In [ ]:
import subprocess, os, shutil

os.chdir('/content/parseq')

# Download base pretrained PARSeq weights
if not os.path.exists('/content/parseq/pretrained/parseq.pt'):
    os.makedirs('/content/parseq/pretrained', exist_ok=True)
    # PARSeq pretrained on synthetic text datasets
    result = subprocess.run([
        'python', '-c',
        'import torch; m = torch.hub.load("baudm/parseq", "parseq", pretrained=True, trust_repo=True); torch.save(m.state_dict(), "/content/parseq/pretrained/parseq.pt"); print("Weights saved")'
    ], capture_output=True, text=True, cwd='/content/parseq')
    print(result.stdout)
    if result.returncode != 0:
        print(f'Weight download issue: {result.stderr[-300:]}')

# Create dataset config for jersey numbers
config_content = f'''
train: /content/crops/positive
val: /content/crops/positive
test: /content/crops/positive
charset_train: '0123456789'
charset_test: '0123456789'
max_label_length: 2
img_h: 32
img_w: 128
'''

os.makedirs('/content/parseq/data', exist_ok=True)
with open('/content/parseq/data/jersey.yaml', 'w') as f:
    f.write(config_content)

# Fine-tune
print('Starting PARSeq fine-tuning...')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}')
print('Expected time on A100: 15-25 minutes')

result = subprocess.run([
    'python', 'train.py',
    'pretrained=parseq',
    f'trainer.max_epochs={EPOCHS}',
    f'data.batch_size={BATCH_SIZE}',
    'data.root_dir=/content/crops',
    'model.charset_train=0123456789',
    'model.charset_test=0123456789',
    'model.max_label_length=2',
], capture_output=True, text=True, cwd='/content/parseq')

print(result.stdout[-1000:])
if result.returncode != 0:
    print(f'Training error: {result.stderr[-500:]}')

# Find best checkpoint
checkpoints = []
for root, dirs, files in os.walk('/content/parseq'):
    for f in files:
        if f.endswith('.ckpt'):
            checkpoints.append(os.path.join(root, f))

if checkpoints:
    latest = max(checkpoints, key=os.path.getmtime)
    print(f'Best checkpoint: {latest}')

    # Save to Drive
    drive_path = f'/content/drive/MyDrive/clipt_parseq_jersey{TARGET_JERSEY}.ckpt'
    shutil.copy(latest, drive_path)
    print(f'Saved to Drive: {drive_path}')
else:
    print('No checkpoints found — training may have failed')

In [ ]:
import torch
from PIL import Image
import cv2, os

# Load fine-tuned model
checkpoint_path = f'/content/drive/MyDrive/clipt_parseq_jersey{TARGET_JERSEY}.ckpt'

if os.path.exists(checkpoint_path):
    print(f'Loading checkpoint: {checkpoint_path}')

    # Quick test on a few crops
    crop_files = os.listdir('/content/crops/positive')[:5]

    model_test = torch.hub.load('baudm/parseq', 'parseq', pretrained=False, trust_repo=True)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    if 'state_dict' in state_dict:
        state_dict = state_dict['state_dict']
    model_test.load_state_dict(state_dict, strict=False)
    model_test.eval()

    print('\nTest predictions on sample crops:')
    for crop_file in crop_files:
        img_path = f'/content/crops/positive/{crop_file}'
        img = Image.open(img_path).convert('RGB')
        # Minimal transform
        from torchvision import transforms
        transform = transforms.Compose([
            transforms.Resize((32, 128)),
            transforms.ToTensor(),
            transforms.Normalize(0.5, 0.5)
        ])
        img_tensor = transform(img).unsqueeze(0)

        with torch.no_grad():
            logits = model_test(img_tensor)
        pred = model_test.tokenizer.decode(logits)
        print(f'  {crop_file}: predicted = "{pred[0]}" (target = {TARGET_JERSEY})')
else:
    print('Checkpoint not found — check training cell output')

In [ ]:
print('''
=== DEPLOYMENT INSTRUCTIONS ===

1. Download the checkpoint from Google Drive:
   /content/drive/MyDrive/clipt_parseq_jersey2.ckpt

2. Upload to your Railway detection server:
   - SCP or upload to app/model/parseq_jersey2.ckpt
   - Or upload to Cloudinary and add the URL to Railway env vars

3. In bytetrack_pipeline.py replace the EasyOCR call in read_jersey_number():

   # OLD (EasyOCR):
   results = _get_ocr().readtext(crop, allowlist=\'0123456789\', detail=1)

   # NEW (PARSeq):
   from .parseq_ocr import read_with_parseq
   number, confidence = read_with_parseq(crop, checkpoint_path=\'app/model/parseq_jersey2.ckpt\')

4. Bump version to v8.34.0 and push to Railway

Expected improvement: 8% jersey hit rate → 70-85%

Colab fine-tune checkpoint saved to:
/content/drive/MyDrive/clipt_parseq_jersey2.ckpt
''')